In [1]:
import rebound
import reboundx
import astropy.constants as constants
import astropy.units as units
import matplotlib.pyplot as plt
import numpy as np
import scipy as sc
from multiprocess import Pool
import celmech

import warnings
from joblib import Parallel, delayed
from celmech.miscellaneous import frequency_modified_fourier_transform as fmftc

In [2]:
def Nintegrate(run, yrs):
    # num = int((yrs-20)*1e6/(3.5e9/1000)) + 2
    num = int((yrs-20)*1e6/(3.5e9/1000)) 
    # print("Snapshot start is," ,num)
    tmax = -yrs*1e6
    tmin = -(yrs-20)*1e6
    Nsnaps = 1000
    interval = int(abs((tmax-tmin)/Nsnaps))
    
    sa = rebound.Simulationarchive("runs/run_" + str(run)+ ".bin")
    orig = sa[0]
    M0 = orig.particles['sun'].m
    earth_m = orig.particles['earth'].m/1.0123000370338813
    
    print("starting ", 'R_'+ str(run)+ '_'+ str(yrs) +'.bin ')
    sim = sa[num]
    sim.integrator_synchronize()

    # print("snapshot time is,", sim.t)
    
    # print("sim min time is,", tmin/1e6, "myr")
    # print("sim max time is,", tmax/1e6, "myr")
    print("sim centered around,", (tmax - ((tmax-tmin)/2))/1e6,"myr")

    sim.integrator = "WHCKL" 
    sim.ri_whfast.safe_mode = False
    sim.ri_whfast.corrector = 17
    sim.ri_whfast.keep_unsynchronized=True
    sim.dt = 4.062/365.25
 
    sim.save_to_file('samples/'+'R_'+ str(run)+ '_'+ str(yrs) +'.bin', interval=interval, delete_file=True)
    
    ps = sim.particles
    rebx = reboundx.Extras(sim)
    gr = rebx.load_force('gr_potential')
    rebx.add_force(gr)
    gr.params['c'] = 63240 # speed of light in AU/yr
    
    cf = rebx.load_force("quadrupole")
    rebx.add_force(cf)

    f = 0.8525
    mu_eff = f*(1*0.0123000370338813*(earth_m)**2)/(1.0123000370338813*earth_m)
    R = 0.0025696
    R_e = (constants.R_earth.to(units.au)).value
    ratio = R / R_e
    r = (ratio-((5.14/ 1.e9)*(abs(sim.t))))*R_e
    ps['earth'].params["Rcentral"] = r
    ps['earth'].params["mu_effcentral"] = mu_eff
    
    gh = rebx.load_force("gravitational_harmonics")
    rebx.add_force(gh)
    J2 = 2.25*1e-7
    sim.particles['sun'].params["J2"] = J2
    
    inc_sun = np.radians(7.155) # rad
    Omega_sun = np.radians(75.594) # rad
    R_eq_sun = (constants.R_sun.to(units.au)).value
    
    spin_axis_vector = [np.sin(inc_sun) * np.sin(Omega_sun), -np.sin(inc_sun) * np.cos(Omega_sun), np.cos(inc_sun)]
    sim.particles['sun'].params["Omega"] = spin_axis_vector
    sim.particles['sun'].params["R_eq"] = R_eq_sun
    
    times = np.linspace(sim.t, tmax, Nsnaps)
    rate = (-7.e14)
    
    Nout = len(sa)

    for i, time in enumerate(times):
        print(sim.t)
        sim.integrate(time)
        sim.particles[0].m = M0*np.exp(time / rate)
        r = (ratio-((5.14/ 1.e9)*(abs(time))))*R_e
        sim.particles['earth'].params["Rcentral"] = r
        sim.move_to_com

In [3]:
# Nintegrate(1, 20)

In [4]:
# def simulation(par):

#     run = par # unpack parameters
#     years = [1190, 3260]

#     for year in years:
#         Nintegrate(run, year)

In [5]:
# %%time 

# with Pool() as pool:
#     Ngrid = 64
#     # Ngrid = 2
#     par_num = np.arange(1,1+Ngrid,1)
#     parameters = []
#     for run in par_num:
#         parameters.append(run)
#     results = pool.map(simulation,parameters)